# Quantum Key Distribution Attack and Security Analysis

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy, chi2
from scipy.optimize import minimize
import hashlib
from collections import defaultdict

class QuantumKeyDistributionAnalyzer:
    def __init__(self, key_length=1024, error_threshold=0.11):
        self.key_length = key_length
        self.error_threshold = error_threshold
        self.bb84_bases = ['rectilinear', 'diagonal']
        self.photon_detector = IsolationForest(contamination=0.1, random_state=42)
        self.scaler = StandardScaler()
        self.attack_signatures = defaultdict(list)
        self.quantum_channel_state = {'noise': 0.01, 'loss': 0.05}

    def simulate_bb84_protocol(self, alice_bits, alice_bases, bob_bases):
        transmitted_photons = []
        for bit, basis in zip(alice_bits, alice_bases):
            polarization = self.encode_photon(bit, basis)
            noisy_photon = self.add_channel_noise(polarization)
            transmitted_photons.append(noisy_photon)

        bob_measurements = []
        for photon, bob_basis in zip(transmitted_photons, bob_bases):
            measured_bit = self.measure_photon(photon, bob_basis)
            bob_measurements.append(measured_bit)

        sifted_key_alice, sifted_key_bob = self.sift_keys(alice_bits, alice_bases, bob_measurements, bob_bases)
        return sifted_key_alice, sifted_key_bob

    def encode_photon(self, bit, basis):
        if basis == 'rectilinear':
            return np.array([1, 0]) if bit == 0 else np.array([0, 1])
        else:
            return np.array([1/np.sqrt(2), 1/np.sqrt(2)]) if bit == 0 else np.array([1/np.sqrt(2), -1/np.sqrt(2)])

    def add_channel_noise(self, photon_state):
        noise_level = self.quantum_channel_state['noise']
        loss_probability = self.quantum_channel_state['loss']

        if np.random.random() < loss_probability:
            return None

        noise = np.random.normal(0, noise_level, photon_state.shape)
        noisy_state = photon_state + noise
        return noisy_state / np.linalg.norm(noisy_state)

    def measure_photon(self, photon_state, measurement_basis):
        if photon_state is None:
            return None

        if measurement_basis == 'rectilinear':
            prob_0 = abs(photon_state[0])**2
        else:
            rotated_state = np.array([photon_state[0]/np.sqrt(2) + photon_state[1]/np.sqrt(2),
                                    photon_state[0]/np.sqrt(2) - photon_state[1]/np.sqrt(2)])
            prob_0 = abs(rotated_state[0])**2

        return 0 if np.random.random() < prob_0 else 1

    def sift_keys(self, alice_bits, alice_bases, bob_bits, bob_bases):
        sifted_alice, sifted_bob = [], []
        for i, (a_base, b_base, a_bit, b_bit) in enumerate(zip(alice_bases, bob_bases, alice_bits, bob_bits)):
            if a_base == b_base and b_bit is not None:
                sifted_alice.append(a_bit)
                sifted_bob.append(b_bit)
        return sifted_alice, sifted_bob

    def intercept_resend_attack(self, alice_bits, alice_bases, bob_bases, eavesdrop_probability=0.5):
        eve_measurements = []
        intercepted_photons = []

        for i, (bit, alice_basis) in enumerate(zip(alice_bits, alice_bases)):
            if np.random.random() < eavesdrop_probability:
                eve_basis = np.random.choice(self.bb84_bases)
                original_photon = self.encode_photon(bit, alice_basis)
                eve_measurement = self.measure_photon(original_photon, eve_basis)
                eve_measurements.append((i, eve_measurement, eve_basis))

                resent_photon = self.encode_photon(eve_measurement, eve_basis)
                intercepted_photons.append((i, resent_photon))
            else:
                intercepted_photons.append((i, self.encode_photon(bit, alice_basis)))

        bob_measurements = []
        for i, (idx, photon) in enumerate(intercepted_photons):
            if i < len(bob_bases):
                measurement = self.measure_photon(photon, bob_bases[i])
                bob_measurements.append(measurement)

        return bob_measurements, eve_measurements

    def photon_number_splitting_attack(self, alice_bits, alice_bases, mean_photon_number=0.1):
        attack_success = []

        for bit, basis in zip(alice_bits, alice_bases):
            photon_count = np.random.poisson(mean_photon_number)

            if photon_count > 1:
                attack_success.append(True)
                stored_photon = self.encode_photon(bit, basis)
                self.attack_signatures['pns'].append({'bit': bit, 'basis': basis, 'photons': photon_count})
            else:
                attack_success.append(False)

        return np.mean(attack_success)

    def quantum_error_correction_analysis(self, alice_key, bob_key):
        if len(alice_key) != len(bob_key):
            return {'error_rate': 1.0, 'correctable': False}

        errors = sum(a != b for a, b in zip(alice_key, bob_key))
        error_rate = errors / len(alice_key)

        syndrome_length = int(np.ceil(np.log2(len(alice_key))))
        correctable_errors = len(alice_key) // (2 ** syndrome_length)

        return {
            'error_rate': error_rate,
            'correctable': errors <= correctable_errors,
            'syndrome_bits': syndrome_length,
            'max_correctable': correctable_errors
        }

    def information_theoretic_security_analysis(self, key_length, error_rate, privacy_amplification_ratio=0.5):
        shannon_entropy = -error_rate * np.log2(error_rate + 1e-10) - (1 - error_rate) * np.log2(1 - error_rate + 1e-10)

        mutual_information_eve = shannon_entropy * key_length
        secure_key_length = key_length - mutual_information_eve - int(key_length * privacy_amplification_ratio)

        return {
            'shannon_entropy': shannon_entropy,
            'eve_information': mutual_information_eve,
            'secure_key_bits': max(0, secure_key_length),
            'security_parameter': -np.log2(2**(-secure_key_length) + 1e-10) if secure_key_length > 0 else 0
        }

    def detect_channel_attacks(self, measurement_statistics):
        features = []

        for stats in measurement_statistics:
            error_patterns = np.array(stats.get('error_positions', []))
            timing_variations = np.array(stats.get('detection_times', []))
            intensity_fluctuations = np.array(stats.get('photon_intensities', []))

            if len(error_patterns) > 0:
                error_clustering = np.std(np.diff(error_patterns)) if len(error_patterns) > 1 else 0
            else:
                error_clustering = 0

            timing_anomaly = np.var(timing_variations) if len(timing_variations) > 0 else 0
            intensity_anomaly = np.var(intensity_fluctuations) if len(intensity_fluctuations) > 0 else 0

            features.append([error_clustering, timing_anomaly, intensity_anomaly])

        if len(features) > 0:
            features_scaled = self.scaler.fit_transform(features)
            anomaly_scores = self.photon_detector.fit_predict(features_scaled)
            return np.mean(anomaly_scores == -1)
        return 0

    def comprehensive_qkd_security_analysis(self, alice_bits, alice_bases, bob_bases):
        clean_alice_key, clean_bob_key = self.simulate_bb84_protocol(alice_bits, alice_bases, bob_bases)

        attacked_bob_measurements, eve_info = self.intercept_resend_attack(alice_bits, alice_bases, bob_bases)
        pns_success_rate = self.photon_number_splitting_attack(alice_bits, alice_bases)

        error_analysis = self.quantum_error_correction_analysis(clean_alice_key, clean_bob_key)
        security_analysis = self.information_theoretic_security_analysis(len(clean_alice_key), error_analysis['error_rate'])

        mock_stats = [{'error_positions': [i for i, (a, b) in enumerate(zip(clean_alice_key, clean_bob_key)) if a != b],
                      'detection_times': np.random.normal(1.0, 0.1, len(clean_alice_key)),
                      'photon_intensities': np.random.exponential(1.0, len(clean_alice_key))}]

        channel_attack_probability = self.detect_channel_attacks(mock_stats)

        overall_security = (error_analysis['error_rate'] < self.error_threshold and
                          security_analysis['secure_key_bits'] > 0 and
                          channel_attack_probability < 0.3)

        return {
            'secure': overall_security,
            'key_length': len(clean_alice_key),
            'error_rate': error_analysis['error_rate'],
            'secure_bits': security_analysis['secure_key_bits'],
            'pns_vulnerability': pns_success_rate,
            'channel_attacks': channel_attack_probability,
            'eve_information': len(eve_info)
        }